In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import torch
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

1
NVIDIA GeForce RTX 4090


In [3]:
import os
import sys
import glob
import warnings
import time
import numpy as np
from scipy import stats, signal
from scipy.io import loadmat
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import logging
import joblib
import cv2

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
from tqdm import tqdm


import torch
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(device)

cuda:1


In [4]:
class Config:
    """Central configuration for the pipeline."""

    # Dataset path (update this to your UTD-MHAD location)
    DATASET_ROOT = './UTD-MHAD Dataset-20251013T122259Z-1-001/UTD-MHAD Dataset'

    # Subdirectories within UTD-MHAD
    SKELETON_DIR = "skeleton"
    INERTIAL_DIR = "inertial"
    DEPTH_DIR = "depth"
    VIDEO_DIRS = ["RGB-part1", "RGB-part2", "RGB-part3", "RGB-part4"]

    # Output directory for saved features (relative to notebook location)
    FEATURES_DIR = "features_new_25_loss"

    # Dataset properties
    NUM_ACTIONS = 27
    NUM_SUBJECTS = 8
    NUM_TRIALS = 4
    NUM_JOINTS = 20  # Kinect v1 skeleton has 20 joints

    # Train/test split: odd subjects for training, even for testing
    # (standard protocol for UTD-MHAD)
    TRAIN_SUBJECTS = [1, 3, 5, 7]
    TEST_SUBJECTS = [2, 4, 6, 8]

    # Inertial feature extraction parameters (inspired by VISTA paper)
    INERTIAL_WINDOW_SIZE = 50     # samples per window
    INERTIAL_WINDOW_OVERLAP = 0.5  # 50% overlap

    # Depth feature extraction parameters
    DEPTH_SAMPLE_FRAMES = 10  # number of frames to sample from each sequence

    # Video (RGB) feature extraction parameters (inspired by VISTA paper)
    VIDEO_SAMPLE_FRAMES = 15   # number of frames to uniformly sample
    VIDEO_RESIZE = (120, 160)  # (height, width) for downsampled frames

    # Random Forest parameters
    RF_N_ESTIMATORS = 300
    RF_MAX_DEPTH = None
    RF_MIN_SAMPLES_SPLIT = 2
    RF_MIN_SAMPLES_LEAF = 1
    RF_RANDOM_STATE = 42
    RF_N_JOBS = -1

    # Whether to use each modality
    USE_SKELETON = True
    USE_INERTIAL = True
    USE_DEPTH = True
    USE_VIDEO = True


In [5]:
# =============================================================================
# SECTION 2: DATA LOADING
# =============================================================================

class UTDMHADLoader:
    """
    Loads the UTD-MHAD dataset from .mat files.

    File naming convention:
        a{action_id}_s{subject_id}_t{trial_id}_{modality}.mat
    """

    def __init__(self, config):
        self.config = config
        self.root = config.DATASET_ROOT

    def _parse_filename(self, filename):
        """Extract action, subject, trial from filename like a1_s1_t1_skeleton.mat"""
        base = os.path.basename(filename)
        parts = base.split('_')
        action = int(parts[0][1:])
        subject = int(parts[1][1:])
        trial = int(parts[2][1:])
        return action, subject, trial

    def load_skeleton_data(self):
        """
        Load skeleton .mat files.
        Each file contains 'd_skel' with shape (frames, 60) = 20 joints × 3 coords
        or shape (frames, 20, 3).
        Returns: list of (action, subject, trial, skeleton_array)
        """
        skeleton_dir = os.path.join(self.root, self.config.SKELETON_DIR)
        files = sorted(glob.glob(os.path.join(skeleton_dir, "a*_s*_t*_skeleton.mat")))

        if not files:
            logger.warning(f"No skeleton files found in {skeleton_dir}")
            return []

        data = []
        for f in files:
            action, subject, trial = self._parse_filename(f)
            try:
                mat = loadmat(f)
                # UTD-MHAD stores skeleton as 'd_skel' with shape (frames, 60)
                skel = mat.get('d_skel', mat.get('d_skeleton', None))
                if skel is None:
                    for key in mat:
                        if not key.startswith('__'):
                            skel = mat[key]
                            break
                if skel is not None:
                    skel = np.array(skel, dtype=np.float64)
                    # Reshape to (frames, 20, 3) if needed
                    if skel.ndim == 2 and skel.shape[1] == 60:
                        skel = skel.reshape(-1, 20, 3)
                    elif skel.ndim == 3 and skel.shape[1] == 20 and skel.shape[2] == 3:
                        pass  # already correct
                    elif skel.ndim == 3 and skel.shape[2] == 20:
                        # shape might be (3, 20, frames) -> transpose
                        skel = skel.transpose(2, 1, 0)
                    data.append((action, subject, trial, skel))
            except Exception as e:
                logger.debug(f"Error loading {f}: {e}")

        logger.info(f"Loaded {len(data)} skeleton sequences")
        return data

    def load_inertial_data(self):
        """
        Load inertial .mat files.
        Each file contains 'd_iner' with shape (samples, 6):
            [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]
        Returns: list of (action, subject, trial, inertial_array)
        """
        inertial_dir = os.path.join(self.root, self.config.INERTIAL_DIR)
        files = sorted(glob.glob(os.path.join(inertial_dir, "a*_s*_t*_inertial.mat")))

        if not files:
            logger.warning(f"No inertial files found in {inertial_dir}")
            return []

        data = []
        for f in files:
            action, subject, trial = self._parse_filename(f)
            try:
                mat = loadmat(f)
                iner = mat.get('d_iner', mat.get('d_inertial', None))
                if iner is None:
                    for key in mat:
                        if not key.startswith('__'):
                            iner = mat[key]
                            break
                if iner is not None:
                    iner = np.array(iner, dtype=np.float64)
                    data.append((action, subject, trial, iner))
            except Exception as e:
                logger.debug(f"Error loading {f}: {e}")

        logger.info(f"Loaded {len(data)} inertial sequences")
        return data

    def load_depth_data(self):
        """
        Load depth .mat files.
        Each file contains 'd_depth' with shape (height, width, frames).
        Returns: list of (action, subject, trial, depth_array)
        """
        depth_dir = os.path.join(self.root, self.config.DEPTH_DIR)
        files = sorted(glob.glob(os.path.join(depth_dir, "a*_s*_t*_depth.mat")))

        if not files:
            logger.warning(f"No depth files found in {depth_dir}")
            return []

        data = []
        for f in files:
            action, subject, trial = self._parse_filename(f)
            try:
                mat = loadmat(f)
                depth = mat.get('d_depth', None)
                if depth is None:
                    for key in mat:
                        if not key.startswith('__'):
                            depth = mat[key]
                            break
                if depth is not None:
                    depth = np.array(depth, dtype=np.float64)
                    data.append((action, subject, trial, depth))
            except Exception as e:
                logger.debug(f"Error loading {f}: {e}")

        logger.info(f"Loaded {len(data)} depth sequences")
        return data

    def load_video_data(self):
        """
        Load RGB video (.avi) files from RGB-part1 through RGB-part4 directories.
        File naming: a{X}_s{Y}_t{Z}_color.avi
        Returns: list of (action, subject, trial, video_filepath)
        """
        files = []
        for vdir in self.config.VIDEO_DIRS:
            video_dir = os.path.join(self.root, vdir)
            found = sorted(glob.glob(os.path.join(video_dir, "a*_s*_t*_color.avi")))
            files.extend(found)

        if not files:
            logger.warning(f"No video files found in {self.config.VIDEO_DIRS}")
            return []

        data = []
        for f in files:
            try:
                action, subject, trial = self._parse_filename(f)
                data.append((action, subject, trial, f))
            except Exception as e:
                logger.debug(f"Error parsing {f}: {e}")

        logger.info(f"Loaded {len(data)} video file paths")
        return data


In [6]:
# =============================================================================
# SECTION 2b: STRUCTURED DATA ZEROING (25% Feature Loss Simulation)
# =============================================================================
# Zero out 25% of **structured** units per modality per sample:
# - **Skeleton**: zero out 25% of joints (joint = all 3 coords across all frames)
# - **Inertial**: zero out 25% of sensor channels (channel = all timesteps for that channel)
# - **Depth**: zero out 25% of frames (frame = entire H×W spatial slice)
# - **Video**: zero out 25% of frames (frame = entire H×W×C slice)
#
# Memory-optimized: operates in-place on arrays already copied into X list,
# avoids redundant .copy() inside zeroing functions, and forces garbage
# collection periodically to prevent OOM crashes on large datasets.
# =============================================================================

import gc
import numpy as np
import random
from tqdm import tqdm

ZERO_FRACTION = 0.25
SEED = 42


def zero_out_skeleton_joints(skeleton_mat, fraction=ZERO_FRACTION, rng=None):
    """
    skeleton_mat: (20 joints, 3 coords, frames)
    Zero out `fraction` of joints — i.e. all coords and all frames for selected joints.
    Operates IN-PLACE (no copy) to save memory.
    """
    n_joints = skeleton_mat.shape[0]
    n_zero = max(1, int(round(n_joints * fraction)))
    joints_to_zero = rng.choice(n_joints, size=n_zero, replace=False)
    skeleton_mat[joints_to_zero, :, :] = 0.0
    return skeleton_mat


def zero_out_inertial_channels(inertial_mat, fraction=ZERO_FRACTION, rng=None):
    """
    inertial_mat: (time_steps, 6 channels)
    Zero out `fraction` of channels — i.e. all timesteps for selected channels.
    Operates IN-PLACE (no copy) to save memory.
    """
    n_channels = inertial_mat.shape[1]
    n_zero = max(1, int(round(n_channels * fraction)))
    channels_to_zero = rng.choice(n_channels, size=n_zero, replace=False)
    inertial_mat[:, channels_to_zero] = 0.0
    return inertial_mat


def zero_out_depth_frames(depth_mat, fraction=ZERO_FRACTION, rng=None):
    """
    depth_mat: (H, W, frames)
    Zero out `fraction` of frames — i.e. entire H×W spatial content for selected frames.
    Operates IN-PLACE (no copy) to save memory.
    """
    n_frames = depth_mat.shape[2]
    n_zero = max(1, int(round(n_frames * fraction)))
    frames_to_zero = rng.choice(n_frames, size=n_zero, replace=False)
    depth_mat[:, :, frames_to_zero] = 0.0
    return depth_mat


def apply_structured_zeroing_to_samples(samples, fraction=ZERO_FRACTION, seed=42):
    rng = np.random.RandomState(seed)
    
    for idx, (key, sample) in enumerate(tqdm(samples.items(), desc="Applying structured zeroing")):
        if 'skeleton' in sample and sample['skeleton'] is not None:
            skel = sample['skeleton']
            
            # Normalize to (frames, 20, 3) if needed
            if skel.ndim == 3:
                if skel.shape[0] == 20 and skel.shape[1] == 3:
                    # (20, 3, frames) -> (frames, 20, 3)
                    skel = np.transpose(skel, (2, 0, 1))
                    sample['skeleton'] = skel
                elif skel.ndim == 2 and skel.shape[1] == 60:
                    skel = skel.reshape(-1, 20, 3)
                    sample['skeleton'] = skel
            
            # Now skel is (frames, 20, 3)
            n_joints = skel.shape[1]
            n_zero = max(1, int(round(n_joints * fraction)))
            joints_to_zero = rng.choice(n_joints, size=n_zero, replace=False)
            skel[:, joints_to_zero, :] = 0.0
        
        if 'inertial' in sample and sample['inertial'] is not None:
            iner = sample['inertial']
            n_ch = iner.shape[1]
            n_zero = max(1, int(round(n_ch * fraction)))
            channels = rng.choice(n_ch, size=n_zero, replace=False)
            iner[:, channels] = 0.0
        
        if 'depth' in sample and sample['depth'] is not None:
            depth = sample['depth']
            n_frames = depth.shape[2]
            n_zero = max(1, int(round(n_frames * fraction)))
            frames_to_zero = rng.choice(n_frames, size=n_zero, replace=False)
            depth[:, :, frames_to_zero] = 0.0
        
        if (idx + 1) % 100 == 0:
            gc.collect()
    
    return samples

In [7]:
# =============================================================================
# SECTION 3: SKELETON FEATURE EXTRACTION
# =============================================================================

class SkeletonFeatureExtractor:
    """
    Extracts features from skeleton joint sequences.

    Inspired by:
    - VISTA paper: normalized joint coordinates relative to torso/hip,
      subset of most discriminative joints
    - MSDFE paper: covariance matrices capturing inter-joint correlations

    Feature categories:
    1. Static pose features (per-frame statistics)
    2. Dynamic motion features (temporal derivatives)
    3. Joint distance features (pairwise distances)
    4. Joint angle features (body part angles)
    5. Covariance features (inter-joint correlation matrix)
    """

    # UTD-MHAD Kinect joint indices (0-indexed):
    # 0: hip center, 1: spine, 2: shoulder center, 3: head,
    # 4: left shoulder, 5: left elbow, 6: left wrist, 7: left hand,
    # 8: right shoulder, 9: right elbow, 10: right wrist, 11: right hand,
    # 12: left hip, 13: left knee, 14: left ankle, 15: left foot,
    # 16: right hip, 17: right knee, 18: right ankle, 19: right foot

    HIP_CENTER = 0
    SPINE = 1
    SHOULDER_CENTER = 2
    HEAD = 3

    # Key joint groups for feature extraction
    UPPER_BODY = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
    LOWER_BODY = [0, 12, 13, 14, 15, 16, 17, 18, 19]
    HANDS = [7, 11]
    FEET = [15, 19]
    ENDPOINTS = [3, 7, 11, 15, 19]  # head, hands, feet

    # Joint pairs for distance features
    JOINT_PAIRS = [
        (7, 11),   # left hand - right hand
        (15, 19),  # left foot - right foot
        (7, 0),    # left hand - hip
        (11, 0),   # right hand - hip
        (7, 3),    # left hand - head
        (11, 3),   # right hand - head
        (15, 0),   # left foot - hip
        (19, 0),   # right foot - hip
        (6, 10),   # left wrist - right wrist
        (5, 9),    # left elbow - right elbow
        (4, 8),    # left shoulder - right shoulder
        (13, 17),  # left knee - right knee
    ]

    # Joint triplets for angle features (vertex is middle joint)
    ANGLE_TRIPLETS = [
        (4, 5, 6),    # left shoulder-elbow-wrist
        (8, 9, 10),   # right shoulder-elbow-wrist
        (12, 13, 14), # left hip-knee-ankle
        (16, 17, 18), # right hip-knee-ankle
        (4, 2, 8),    # left shoulder - shoulder center - right shoulder
        (5, 4, 2),    # left elbow - left shoulder - shoulder center
        (9, 8, 2),    # right elbow - right shoulder - shoulder center
        (0, 1, 2),    # hip - spine - shoulder center
    ]

    def __init__(self):
        pass

    def _normalize_skeleton(self, skel):
        """
        Normalize skeleton: translate to hip center, scale by torso length.
        Inspired by VISTA paper's torso-based normalization.

        Args:
            skel: (frames, 20, 3)
        Returns:
            normalized skeleton (frames, 20, 3)
        """
        # Translate: hip center as origin
        hip = skel[:, self.HIP_CENTER:self.HIP_CENTER+1, :]  # (frames, 1, 3)
        skel_norm = skel - hip

        # Scale: normalize by torso length (hip to shoulder center)
        torso_vec = skel_norm[:, self.SHOULDER_CENTER, :] - skel_norm[:, self.HIP_CENTER, :]
        torso_len = np.linalg.norm(torso_vec, axis=1, keepdims=True)  # (frames, 1)
        torso_len = np.clip(torso_len, 1e-6, None)
        skel_norm = skel_norm / torso_len[:, np.newaxis, :]

        return skel_norm

    def _compute_joint_distances(self, skel):
        """Compute pairwise joint distances over time."""
        distances = []
        for j1, j2 in self.JOINT_PAIRS:
            dist = np.linalg.norm(skel[:, j1, :] - skel[:, j2, :], axis=1)
            distances.append(dist)
        return np.array(distances).T  # (frames, num_pairs)

    def _compute_joint_angles(self, skel):
        """Compute angles at joint triplets using dot product."""
        angles = []
        for j1, j2, j3 in self.ANGLE_TRIPLETS:
            v1 = skel[:, j1, :] - skel[:, j2, :]
            v2 = skel[:, j3, :] - skel[:, j2, :]
            cos_angle = np.sum(v1 * v2, axis=1) / (
                np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-8
            )
            cos_angle = np.clip(cos_angle, -1.0, 1.0)
            angle = np.arccos(cos_angle)
            angles.append(angle)
        return np.array(angles).T  # (frames, num_angles)

    def _compute_velocity(self, skel):
        """Compute first-order temporal derivatives (velocity)."""
        if skel.shape[0] < 2:
            return np.zeros_like(skel)
        vel = np.diff(skel, axis=0)
        vel = np.vstack([vel, vel[-1:]])  # pad to same length
        return vel

    def _compute_acceleration(self, skel):
        """Compute second-order temporal derivatives (acceleration)."""
        vel = self._compute_velocity(skel)
        if vel.shape[0] < 2:
            return np.zeros_like(vel)
        acc = np.diff(vel, axis=0)
        acc = np.vstack([acc, acc[-1:]])
        return acc

    def _temporal_statistics(self, signal_2d):
        """
        Compute temporal statistics over a 2D signal (frames, features).
        Inspired by VISTA paper's feature set.

        Returns: 1D feature vector with stats per feature column
        """
        features = []
        for col in range(signal_2d.shape[1]):
            s = signal_2d[:, col]
            features.extend([
                np.mean(s),
                np.std(s),
                np.sqrt(np.mean(s**2)),           # RMS
                stats.skew(s) if len(s) > 2 else 0.0,
                stats.kurtosis(s) if len(s) > 3 else 0.0,
                np.max(s) - np.min(s),            # range
                np.median(s),
                np.percentile(s, 25),             # Q1
                np.percentile(s, 75),             # Q3
            ])
        return np.array(features)

    def _covariance_features(self, skel):
        """
        Compute covariance matrix features (inspired by MSDFE paper).
        The covariance matrix captures inter-joint spatial correlations.

        Args:
            skel: (frames, 20, 3) normalized skeleton
        Returns:
            Upper triangular elements of the covariance matrix
        """
        # Flatten joints: (frames, 60)
        flat = skel.reshape(skel.shape[0], -1)

        # Select a subset of key joints to keep feature size manageable
        # Use endpoints + torso: head, hands, feet, hip, spine, shoulders
        key_joints = [0, 1, 2, 3, 7, 11, 15, 19]
        key_indices = []
        for j in key_joints:
            key_indices.extend([j*3, j*3+1, j*3+2])

        flat_key = flat[:, key_indices]  # (frames, 24)

        # Compute covariance matrix
        if flat_key.shape[0] < 2:
            cov_mat = np.zeros((flat_key.shape[1], flat_key.shape[1]))
        else:
            cov_mat = np.cov(flat_key.T)

        # Extract upper triangular elements (including diagonal)
        upper_tri = cov_mat[np.triu_indices(cov_mat.shape[0])]
        return upper_tri

    def extract(self, skel):
        """
        Extract full skeleton feature vector from a single sequence.

        Args:
            skel: (frames, 20, 3) skeleton data
        Returns:
            1D feature vector
        """
        if skel is None or skel.shape[0] == 0:
            return None
        
        # Convert from (20, 3, frames) → (frames, 20, 3)
        if skel.ndim != 3:
            return None
        
        # Accept either (frames, 20, 3) or (20, 3, frames)
        if skel.shape[0] == 20 and skel.shape[1] == 3:
            skel = np.transpose(skel, (2, 0, 1))  # -> (frames, 20, 3)
        elif skel.shape[1] == 20 and skel.shape[2] == 3:
            pass  # already (frames, 20, 3)
        else:
            skel = np.transpose(skel, (2, 0, 1))  # fallback, original behavior

        skel = np.nan_to_num(skel, nan=0.0, posinf=0.0, neginf=0.0)
        

        # 1. Normalize
        skel_norm = self._normalize_skeleton(skel)

        # 2. Joint positions - temporal statistics on all joint coords
        flat_pos = skel_norm.reshape(skel_norm.shape[0], -1)  # (frames, 60)
        # Use subset of key joints for position stats
        key_joint_ids = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 15, 19]
        key_pos_idx = []
        for j in key_joint_ids:
            key_pos_idx.extend([j*3, j*3+1, j*3+2])
        pos_feats = self._temporal_statistics(flat_pos[:, key_pos_idx])

        # 3. Joint distances over time
        distances = self._compute_joint_distances(skel_norm)
        dist_feats = self._temporal_statistics(distances)

        # 4. Joint angles over time
        angles = self._compute_joint_angles(skel_norm)
        angle_feats = self._temporal_statistics(angles)

        # 5. Velocity features
        velocity = self._compute_velocity(skel_norm)
        vel_flat = velocity.reshape(velocity.shape[0], -1)
        # Key joints velocity
        vel_key = vel_flat[:, key_pos_idx]
        vel_feats = self._temporal_statistics(vel_key)

        # 6. Acceleration features
        accel = self._compute_acceleration(skel_norm)
        acc_flat = accel.reshape(accel.shape[0], -1)
        acc_key = acc_flat[:, key_pos_idx]
        acc_feats = self._temporal_statistics(acc_key)

        # 7. Covariance features (MSDFE-inspired)
        cov_feats = self._covariance_features(skel_norm)

        # 8. Global motion features
        # Center of mass trajectory
        com = np.mean(skel_norm, axis=1)  # (frames, 3)
        com_feats = self._temporal_statistics(com)

        # Total body movement (sum of all joint displacements)
        if skel_norm.shape[0] > 1:
            total_disp = np.sum(np.linalg.norm(np.diff(skel_norm, axis=0), axis=2), axis=1)
        else:
            total_disp = np.array([0.0])
        motion_energy = np.array([
            np.mean(total_disp), np.std(total_disp), np.max(total_disp),
            np.sum(total_disp)
        ])

        # Concatenate all skeleton features
        all_feats = np.concatenate([
            pos_feats,      # Position statistics
            dist_feats,     # Distance statistics
            angle_feats,    # Angle statistics
            vel_feats,      # Velocity statistics
            acc_feats,      # Acceleration statistics
            cov_feats,      # Covariance matrix features
            com_feats,      # Center of mass
            motion_energy,  # Global motion
        ])

        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)


In [8]:
# =============================================================================
# SECTION 4: INERTIAL FEATURE EXTRACTION
# =============================================================================

class InertialFeatureExtractor:
    """
    Extracts features from inertial sensor data (accelerometer + gyroscope).

    Directly inspired by the VISTA paper (Fiorini et al., 2022):
    - Time-domain features: mean, std, RMS, skewness, kurtosis, SMA, power
    - Additional: zero-crossing rate, peak count, signal energy
    - Frequency-domain: dominant frequency, spectral entropy, band energy

    The VISTA paper demonstrated these features on wrist + finger IMUs
    and showed 73-81% accuracy with individual sensors and higher with fusion.
    """

    def __init__(self, config):
        self.window_size = config.INERTIAL_WINDOW_SIZE
        self.overlap = config.INERTIAL_WINDOW_OVERLAP

    def _signal_magnitude_area(self, data):
        """
        Signal Magnitude Area (SMA) - used in VISTA paper.
        SMA = (1/N) * sum(|ax| + |ay| + |az|)
        """
        return np.mean(np.sum(np.abs(data), axis=1))

    def _signal_power(self, s):
        """Signal power = mean of squared values."""
        return np.mean(s**2)

    def _zero_crossing_rate(self, s):
        """Count zero crossings normalized by length."""
        s_centered = s - np.mean(s)
        return np.sum(np.abs(np.diff(np.sign(s_centered)))) / (2.0 * len(s))

    def _peak_count(self, s):
        """Count number of peaks in signal."""
        peaks, _ = signal.find_peaks(s)
        return len(peaks) / len(s)

    def _spectral_entropy(self, s, fs=50):
        """Compute spectral entropy of the signal."""
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        psd_norm = psd / (np.sum(psd) + 1e-12)
        psd_norm = psd_norm[psd_norm > 0]
        return -np.sum(psd_norm * np.log2(psd_norm + 1e-12))

    def _dominant_frequency(self, s, fs=50):
        """Find the dominant frequency component."""
        if len(s) < 4:
            return 0.0
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        return freqs[np.argmax(psd)]

    def _frequency_band_energy(self, s, fs=50, bands=[(0, 5), (5, 15), (15, 25)]):
        """Energy in different frequency bands."""
        if len(s) < 4:
            return [0.0] * len(bands)
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        energies = []
        for low, high in bands:
            mask = (freqs >= low) & (freqs < high)
            energies.append(np.sum(psd[mask]))
        return energies

    def _extract_channel_features(self, s):
        """
        Extract comprehensive features from a single channel.
        Based on VISTA paper's feature set + extensions.
        """
        features = []

        # Time-domain (VISTA paper features)
        features.append(np.mean(s))                          # Mean
        features.append(np.std(s))                           # Standard deviation
        features.append(np.sqrt(np.mean(s**2)))              # RMS
        features.append(stats.skew(s) if len(s) > 2 else 0) # Skewness
        features.append(stats.kurtosis(s) if len(s) > 3 else 0)  # Kurtosis
        features.append(self._signal_power(s))               # Power
        features.append(np.max(s) - np.min(s))               # Range
        features.append(np.median(s))                        # Median
        features.append(np.mean(np.abs(s)))                  # Mean absolute value
        features.append(np.percentile(s, 25))                # Q1
        features.append(np.percentile(s, 75))                # Q3
        features.append(np.percentile(s, 75) - np.percentile(s, 25))  # IQR
        features.append(self._zero_crossing_rate(s))         # Zero-crossing rate
        features.append(self._peak_count(s))                 # Peak count

        # Frequency-domain
        features.append(self._dominant_frequency(s))         # Dominant freq
        features.append(self._spectral_entropy(s))           # Spectral entropy
        features.extend(self._frequency_band_energy(s))      # Band energies

        return features

    def _extract_window_features(self, window):
        """
        Extract features from a single window of inertial data.

        Args:
            window: (window_size, 6) array [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]
        Returns:
            1D feature vector
        """
        features = []

        acc = window[:, :3]   # Accelerometer
        gyro = window[:, 3:]  # Gyroscope

        # Per-channel features for all 6 channels
        for ch in range(6):
            features.extend(self._extract_channel_features(window[:, ch]))

        # Accelerometer magnitude
        acc_mag = np.linalg.norm(acc, axis=1)
        features.extend(self._extract_channel_features(acc_mag))

        # Gyroscope magnitude
        gyro_mag = np.linalg.norm(gyro, axis=1)
        features.extend(self._extract_channel_features(gyro_mag))

        # Signal Magnitude Area (SMA) - key VISTA feature
        features.append(self._signal_magnitude_area(acc))
        features.append(self._signal_magnitude_area(gyro))

        # Cross-axis correlations (inspired by MSDFE covariance idea)
        for i in range(3):
            for j in range(i+1, 3):
                if len(acc[:, i]) > 1:
                    corr = np.corrcoef(acc[:, i], acc[:, j])[0, 1]
                    features.append(corr if not np.isnan(corr) else 0.0)
                else:
                    features.append(0.0)

        for i in range(3):
            for j in range(i+1, 3):
                if len(gyro[:, i]) > 1:
                    corr = np.corrcoef(gyro[:, i], gyro[:, j])[0, 1]
                    features.append(corr if not np.isnan(corr) else 0.0)
                else:
                    features.append(0.0)

        # Acc-Gyro cross-correlations
        for i in range(3):
            if len(acc[:, i]) > 1:
                corr = np.corrcoef(acc[:, i], gyro[:, i])[0, 1]
                features.append(corr if not np.isnan(corr) else 0.0)
            else:
                features.append(0.0)

        return features

    def extract(self, inertial_data):
        """
        Extract feature vector from a full inertial sequence.

        Strategy: segment into overlapping windows, extract features per window,
        then aggregate window features with statistics.

        Args:
            inertial_data: (samples, 6) array
        Returns:
            1D feature vector
        """
        if inertial_data is None or inertial_data.shape[0] == 0:
            return None

        inertial_data = np.nan_to_num(inertial_data, nan=0.0, posinf=0.0, neginf=0.0)

        # Apply low-pass Butterworth filter (VISTA paper uses 5Hz cutoff)
        try:
            b, a = signal.butter(4, 0.2, btype='low')  # Normalized frequency
            for ch in range(inertial_data.shape[1]):
                if len(inertial_data[:, ch]) > 12:
                    inertial_data[:, ch] = signal.filtfilt(b, a, inertial_data[:, ch])
        except Exception:
            pass  # Skip filtering if it fails

        # Segment into windows
        n_samples = inertial_data.shape[0]
        step = int(self.window_size * (1 - self.overlap))
        step = max(step, 1)

        windows = []
        for start in range(0, n_samples - self.window_size + 1, step):
            windows.append(inertial_data[start:start + self.window_size])

        if not windows:
            # If sequence is shorter than window, use entire sequence
            windows = [inertial_data]

        # Extract features from each window
        window_features = []
        for w in windows:
            wf = self._extract_window_features(w)
            window_features.append(wf)

        window_features = np.array(window_features)

        # Aggregate: compute statistics across windows
        if window_features.shape[0] == 1:
            return np.nan_to_num(window_features[0], nan=0.0, posinf=0.0, neginf=0.0)

        aggregated = []
        for col in range(window_features.shape[1]):
            col_data = window_features[:, col]
            aggregated.extend([
                np.mean(col_data),
                np.std(col_data),
                np.min(col_data),
                np.max(col_data),
            ])

        return np.nan_to_num(np.array(aggregated), nan=0.0, posinf=0.0, neginf=0.0)


In [9]:
# =============================================================================
# SECTION 5: DEPTH FEATURE EXTRACTION
# =============================================================================

class DepthFeatureExtractor:
    """
    Extracts features from depth map sequences.

    Inspired by:
    - Siddiqi & Alrashdi (2022): Edge detection features from depth maps
      using Sobel operators, edge magnitude/direction histograms
    - MSDFE paper: Covariance-based statistical features from spatial regions

    Feature categories:
    1. Silhouette shape features (from thresholded depth)
    2. Edge-based features (Sobel gradients, inspired by Paper 3)
    3. Depth statistics (distribution of depth values)
    4. Temporal motion features (frame differences)
    5. HOG-like gradient histograms
    """

    def __init__(self, config):
        self.n_sample_frames = config.DEPTH_SAMPLE_FRAMES

    def _get_silhouette(self, depth_frame):
        """Extract binary silhouette from depth frame."""
        # Person is the closest non-zero depth
        valid = depth_frame[depth_frame > 0]
        if len(valid) == 0:
            return np.zeros_like(depth_frame, dtype=bool)
        threshold = np.percentile(valid, 50)
        return (depth_frame > 0) & (depth_frame < threshold)

    def _sobel_features(self, frame):
        """
        Compute Sobel edge features (inspired by Siddiqi paper).
        Edge magnitude and direction histograms.
        """
        # Sobel operators
        sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
        sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)

        from scipy.ndimage import convolve
        gx = convolve(frame.astype(np.float64), sobel_x)
        gy = convolve(frame.astype(np.float64), sobel_y)

        # Edge magnitude
        magnitude = np.sqrt(gx**2 + gy**2)
        # Edge direction
        direction = np.arctan2(gy, gx + 1e-12)

        features = []

        # Magnitude statistics
        features.extend([
            np.mean(magnitude),
            np.std(magnitude),
            np.max(magnitude),
            np.sum(magnitude > np.mean(magnitude)),  # edge pixel count
        ])

        # Direction histogram (8 bins, like HOG)
        dir_hist, _ = np.histogram(direction[magnitude > np.mean(magnitude)],
                                    bins=8, range=(-np.pi, np.pi))
        if np.sum(dir_hist) > 0:
            dir_hist = dir_hist / (np.sum(dir_hist) + 1e-12)
        features.extend(dir_hist.tolist())

        # Gradient magnitude histogram
        mag_hist, _ = np.histogram(magnitude.ravel(), bins=8)
        if np.sum(mag_hist) > 0:
            mag_hist = mag_hist / (np.sum(mag_hist) + 1e-12)
        features.extend(mag_hist.tolist())

        return features

    def _shape_features(self, silhouette):
        """Compute shape descriptors from binary silhouette."""
        features = []

        area = np.sum(silhouette)
        features.append(area)

        if area == 0:
            return features + [0] * 7

        # Bounding box
        rows = np.any(silhouette, axis=1)
        cols = np.any(silhouette, axis=0)
        if np.any(rows) and np.any(cols):
            rmin, rmax = np.where(rows)[0][[0, -1]]
            cmin, cmax = np.where(cols)[0][[0, -1]]
            height = rmax - rmin + 1
            width = cmax - cmin + 1
            features.append(height)
            features.append(width)
            features.append(height / (width + 1e-6))  # aspect ratio
            features.append(area / (height * width + 1e-6))  # extent
        else:
            features.extend([0, 0, 0, 0])

        # Centroid
        y_coords, x_coords = np.where(silhouette)
        if len(y_coords) > 0:
            features.append(np.mean(y_coords) / silhouette.shape[0])
            features.append(np.mean(x_coords) / silhouette.shape[1])
            features.append(np.std(y_coords) / (silhouette.shape[0] + 1e-6))
        else:
            features.extend([0, 0, 0])

        return features

    def _depth_distribution_features(self, depth_frame):
        """Statistical features from depth value distribution."""
        valid = depth_frame[depth_frame > 0].ravel()
        if len(valid) == 0:
            return [0] * 8

        features = [
            np.mean(valid),
            np.std(valid),
            np.median(valid),
            np.min(valid),
            np.max(valid),
            stats.skew(valid) if len(valid) > 2 else 0,
            stats.kurtosis(valid) if len(valid) > 3 else 0,
            np.percentile(valid, 75) - np.percentile(valid, 25),  # IQR
        ]
        return features

    def _extract_frame_features(self, frame):
        """Extract features from a single depth frame."""
        features = []

        # Resize/downsample for efficiency
        h, w = frame.shape
        scale = 4
        small = frame[::scale, ::scale]

        # Silhouette features
        sil = self._get_silhouette(small)
        features.extend(self._shape_features(sil))

        # Edge features (Siddiqi-inspired)
        features.extend(self._sobel_features(small))

        # Depth distribution
        features.extend(self._depth_distribution_features(small))

        return features

    def extract(self, depth_data):
        """
        Extract feature vector from a depth sequence.

        Args:
            depth_data: (height, width, frames) or (frames, height, width)
        Returns:
            1D feature vector
        """
        if depth_data is None or depth_data.size == 0:
            return None

        depth_data = np.nan_to_num(depth_data, nan=0.0, posinf=0.0, neginf=0.0)

        # Ensure shape is (frames, height, width)
        if depth_data.ndim == 3:
            if depth_data.shape[0] > depth_data.shape[2]:
                # Likely (height, width, frames) -> transpose
                depth_data = np.transpose(depth_data, (2, 0, 1))
        else:
            return None

        n_frames = depth_data.shape[0]

        # Sample frames uniformly
        if n_frames <= self.n_sample_frames:
            frame_indices = list(range(n_frames))
        else:
            frame_indices = np.linspace(0, n_frames - 1, self.n_sample_frames, dtype=int)

        # Extract per-frame features
        frame_features = []
        for idx in frame_indices:
            ff = self._extract_frame_features(depth_data[idx])
            frame_features.append(ff)

        frame_features = np.array(frame_features)

        # Temporal features from frame differences
        temporal_feats = []
        if frame_features.shape[0] > 1:
            diffs = np.diff(frame_features, axis=0)
            for col in range(diffs.shape[1]):
                temporal_feats.extend([
                    np.mean(diffs[:, col]),
                    np.std(diffs[:, col]),
                ])
        else:
            temporal_feats = [0.0] * (frame_features.shape[1] * 2)

        # Aggregate frame features
        aggregated = []
        for col in range(frame_features.shape[1]):
            col_data = frame_features[:, col]
            aggregated.extend([
                np.mean(col_data),
                np.std(col_data),
                np.min(col_data),
                np.max(col_data),
            ])

        all_feats = np.concatenate([aggregated, temporal_feats])
        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)


In [10]:
# =============================================================================
# SECTION 5b: VIDEO (RGB) FEATURE EXTRACTION
# =============================================================================

import cv2
import numpy as np
from scipy import stats
import mediapipe as mp
import logging

logger = logging.getLogger(__name__)


class VideoFeatureExtractor:
    """
    Extracts features from RGB video sequences.

    Directly inspired by the VISTA paper (Fiorini et al., 2022):
    - Extracts 2D skeleton keypoints from video frames using MediaPipe Pose
    - Normalizes 2D joint coordinates relative to the torso (hip/shoulder
      midpoint), following VISTA's torso-based normalization
    - Selects a discriminative subset of body keypoints (head, neck/torso,
      hands/wrists, feet/ankles)
    - Segments into overlapping temporal windows and computes mean joint
      positions per window (VISTA used 3-second windows with 50% overlap)
    - Extracts temporal statistics: mean, std, RMS, skewness, kurtosis,
      range, median, Q1, Q3 over the full joint trajectory sequence

    Additional video-specific features beyond VISTA:
    - Optical flow magnitude/direction statistics for motion encoding
    - Frame-level appearance features: color histograms, HOG-like
      gradient descriptors from person bounding box regions
    - Temporal dynamics: velocity/acceleration of 2D keypoints

    Feature categories:
    1. 2D pose keypoints with torso normalization (VISTA core method)
    2. Keypoint velocity and acceleration over time
    3. Pairwise keypoint distances (hand-hand, hand-head, etc.)
    4. Optical flow motion features (magnitude + direction stats)
    5. Appearance features: color histograms from person region
    6. Covariance of joint trajectories (MSDFE-inspired)
    """

    # MediaPipe Pose landmark indices mapped to our 14-keypoint skeleton:
    # 0: head (nose)           -> mp landmark 0
    # 1: neck (midpoint of shoulders, computed)
    # 2: left_shoulder         -> mp landmark 11
    # 3: right_shoulder        -> mp landmark 12
    # 4: left_elbow            -> mp landmark 13
    # 5: right_elbow           -> mp landmark 14
    # 6: left_wrist            -> mp landmark 15
    # 7: right_wrist           -> mp landmark 16
    # 8: left_hip              -> mp landmark 23
    # 9: right_hip             -> mp landmark 24
    # 10: left_knee            -> mp landmark 25
    # 11: right_knee           -> mp landmark 26
    # 12: left_ankle           -> mp landmark 27
    # 13: right_ankle          -> mp landmark 28
    NUM_KEYPOINTS = 14

    # Mapping from our keypoint index to MediaPipe Pose landmark index
    # (neck=1 is computed as midpoint of shoulders, not directly mapped)
    MP_LANDMARK_MAP = {
        0: 0,    # nose -> head
        2: 11,   # left shoulder
        3: 12,   # right shoulder
        4: 13,   # left elbow
        5: 14,   # right elbow
        6: 15,   # left wrist
        7: 16,   # right wrist
        8: 23,   # left hip
        9: 24,   # right hip
        10: 25,  # left knee
        11: 26,  # right knee
        12: 27,  # left ankle
        13: 28,  # right ankle
    }

    # Keypoint pairs for distance features (same as original)
    KEYPOINT_PAIRS = [
        (6, 7),    # left wrist - right wrist
        (12, 13),  # left ankle - right ankle
        (6, 0),    # left wrist - head
        (7, 0),    # right wrist - head
        (6, 8),    # left wrist - left hip
        (7, 9),    # right wrist - right hip
        (2, 3),    # left shoulder - right shoulder
        (8, 9),    # left hip - right hip
    ]

    def __init__(self, config):
        self.n_sample_frames = config.VIDEO_SAMPLE_FRAMES
        self.resize = config.VIDEO_RESIZE

        # Initialize MediaPipe Pose
        self.mp_pose = mp.solutions.pose
        self.pose = self.mp_pose.Pose(
            static_image_mode=False,
            model_complexity=1,       # 0=lite, 1=full, 2=heavy
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )

    def __del__(self):
        """Clean up MediaPipe resources."""
        if hasattr(self, 'pose'):
            self.pose.close()

    def _read_video_frames(self, video_path):
        """Read and sample frames from a video file."""
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            logger.debug(f"Cannot open video: {video_path}")
            return None

        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)
        cap.release()

        if len(frames) == 0:
            return None

        # Uniformly sample frames
        n = len(frames)
        if n <= self.n_sample_frames:
            indices = list(range(n))
        else:
            indices = np.linspace(0, n - 1, self.n_sample_frames, dtype=int)

        sampled = [frames[i] for i in indices]

        # Resize for efficiency
        resized = []
        for f in sampled:
            r = cv2.resize(f, (self.resize[1], self.resize[0]))
            resized.append(r)

        return resized

    def _estimate_keypoints_mediapipe(self, frame):
        """
        Estimate 2D keypoints using MediaPipe Pose.

        Returns:
            keypoints: (14, 2) array of (x, y) normalized to [0, 1]
            bbox: (y_min, y_max, x_min, x_max) bounding box or None
            visibility: (14,) array of visibility scores
        """
        h, w = frame.shape[:2]

        # MediaPipe expects RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.pose.process(rgb_frame)

        keypoints = np.zeros((self.NUM_KEYPOINTS, 2))
        visibility = np.zeros(self.NUM_KEYPOINTS)
        bbox = None

        if results.pose_landmarks is not None:
            landmarks = results.pose_landmarks.landmark

            # Map MediaPipe landmarks to our 14-keypoint skeleton
            for our_idx, mp_idx in self.MP_LANDMARK_MAP.items():
                lm = landmarks[mp_idx]
                keypoints[our_idx] = [lm.x, lm.y]  # already normalized [0, 1]
                visibility[our_idx] = lm.visibility

            # Neck (index 1) = midpoint of left shoulder and right shoulder
            lm_ls = landmarks[11]
            lm_rs = landmarks[12]
            keypoints[1] = [
                (lm_ls.x + lm_rs.x) / 2.0,
                (lm_ls.y + lm_rs.y) / 2.0,
            ]
            visibility[1] = min(lm_ls.visibility, lm_rs.visibility)

            # Compute bounding box from all mapped landmarks
            all_x = [keypoints[i, 0] * w for i in range(self.NUM_KEYPOINTS)]
            all_y = [keypoints[i, 1] * h for i in range(self.NUM_KEYPOINTS)]
            margin = 0.05  # 5% margin
            x_min = max(0, int(min(all_x) - margin * w))
            x_max = min(w, int(max(all_x) + margin * w))
            y_min = max(0, int(min(all_y) - margin * h))
            y_max = min(h, int(max(all_y) + margin * h))

            if x_max > x_min and y_max > y_min:
                bbox = (y_min, y_max, x_min, x_max)

        return keypoints, bbox, visibility

    def _normalize_keypoints(self, keypoints_seq):
        """
        VISTA-style torso normalization for 2D keypoints.
        Translate to torso center, scale by torso size.

        Args:
            keypoints_seq: (frames, 14, 2)
        Returns:
            normalized (frames, 14, 2)
        """
        # Torso center = midpoint of (neck, left_hip, right_hip)
        torso_center = np.mean(keypoints_seq[:, [1, 8, 9], :], axis=1, keepdims=True)
        kp_norm = keypoints_seq - torso_center

        # Scale by shoulder-hip distance (torso size)
        shoulder_mid = np.mean(keypoints_seq[:, [2, 3], :], axis=1)
        hip_mid = np.mean(keypoints_seq[:, [8, 9], :], axis=1)
        torso_len = np.linalg.norm(shoulder_mid - hip_mid, axis=1, keepdims=True)
        torso_len = np.clip(torso_len, 1e-6, None)
        kp_norm = kp_norm / torso_len[:, np.newaxis, :]

        return kp_norm

    def _temporal_statistics(self, signal_2d):
        """
        Compute temporal statistics per feature column.
        Same stats as VISTA paper: mean, std, RMS, skewness, kurtosis,
        range, median, Q1, Q3.
        """
        features = []
        for col in range(signal_2d.shape[1]):
            s = signal_2d[:, col]
            features.extend([
                np.mean(s),
                np.std(s),
                np.sqrt(np.mean(s**2)),
                stats.skew(s) if len(s) > 2 else 0.0,
                stats.kurtosis(s) if len(s) > 3 else 0.0,
                np.max(s) - np.min(s),
                np.median(s),
                np.percentile(s, 25),
                np.percentile(s, 75),
            ])
        return np.array(features)

    def _compute_optical_flow(self, frames):
        """
        Compute dense optical flow between consecutive frames.
        Returns per-frame flow magnitude and direction statistics.
        """
        if len(frames) < 2:
            return np.zeros((1, 8))

        flow_features = []
        prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)

        for i in range(1, len(frames)):
            curr_gray = cv2.cvtColor(frames[i], cv2.COLOR_BGR2GRAY)
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, curr_gray, None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

            flow_features.append([
                np.mean(mag),
                np.std(mag),
                np.max(mag),
                np.sum(mag > np.mean(mag)),  # active pixel count
                np.mean(ang),
                np.std(ang),
                # Circular histogram entropy of direction (4 quadrants)
                *[np.sum((ang >= q * np.pi / 2) & (ang < (q + 1) * np.pi / 2))
                  / (mag.size + 1e-12) for q in range(4)]
            ])
            prev_gray = curr_gray

        return np.array(flow_features)  # (n_frames-1, 10)

    def _color_histogram_features(self, frames, bbox_list):
        """
        Extract color histogram features from person region.
        Inspired by VISTA's use of visual appearance information.
        """
        hist_features = []
        for frame, bbox in zip(frames, bbox_list):
            if bbox is None:
                hist_features.append(np.zeros(48))
                continue
            y1, y2, x1, x2 = bbox
            roi = frame[y1:y2, x1:x2]
            if roi.size == 0:
                hist_features.append(np.zeros(48))
                continue

            hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
            # H: 16 bins, S: 16 bins, V: 16 bins
            h_hist = cv2.calcHist([hsv], [0], None, [16], [0, 180]).ravel()
            s_hist = cv2.calcHist([hsv], [1], None, [16], [0, 256]).ravel()
            v_hist = cv2.calcHist([hsv], [2], None, [16], [0, 256]).ravel()

            total = h_hist.sum() + 1e-12
            h_hist /= total
            s_hist /= total
            v_hist /= total

            hist_features.append(np.concatenate([h_hist, s_hist, v_hist]))

        return np.array(hist_features)  # (n_frames, 48)

    def extract(self, video_path, apply_zeroing=False, zero_fraction=0.10, zero_rng=None):
        """
        Extract VISTA-inspired feature vector from an RGB video.

        Pipeline:
        1. Read & sample frames
        1b. (Optional) Apply structured zeroing to 10% of frames
        2. Estimate 2D keypoints per frame using MediaPipe Pose
        3. Normalize keypoints to torso (VISTA method)
        4. Compute temporal statistics on keypoint trajectories
        5. Compute optical flow motion features
        6. Compute color histogram appearance features
        7. Concatenate all features

        Args:
            video_path: path to .avi video file
            apply_zeroing: if True, zero out a fraction of video frames
            zero_fraction: fraction of frames to zero out (default 0.10)
            zero_rng: numpy RandomState for reproducibility
        Returns:
            1D feature vector or None
        """
        frames = self._read_video_frames(video_path)
        if frames is None or len(frames) < 2:
            return None

        # Apply structured zeroing to video frames if requested
        if apply_zeroing and zero_rng is not None:
            n_frames = len(frames)
            n_zero = max(1, int(round(n_frames * zero_fraction)))
            frames_to_zero = zero_rng.choice(n_frames, size=n_zero, replace=False)
            for idx in frames_to_zero:
                frames[idx] = np.zeros_like(frames[idx])

        # --- Step 1: Estimate keypoints per frame using MediaPipe ---
        keypoints_seq = []
        bboxes = []
        visibility_seq = []

        for frame in frames:
            kp, bbox, vis = self._estimate_keypoints_mediapipe(frame)
            keypoints_seq.append(kp)
            visibility_seq.append(vis)

            if bbox is not None:
                bboxes.append(bbox)
            else:
                # Fallback bbox: central region of frame
                h, w = frame.shape[:2]
                bboxes.append((h // 4, 3 * h // 4, w // 4, 3 * w // 4))

        keypoints_seq = np.array(keypoints_seq)      # (n_frames, 14, 2)
        visibility_seq = np.array(visibility_seq)      # (n_frames, 14)

        # --- Step 1b: Interpolate low-visibility keypoints ---
        # If MediaPipe failed on some frames, interpolate from neighbors
        for kp_idx in range(self.NUM_KEYPOINTS):
            vis = visibility_seq[:, kp_idx]
            low_vis_mask = vis < 0.3
            if low_vis_mask.all():
                # No reliable detections for this keypoint; leave as-is
                continue
            if low_vis_mask.any() and not low_vis_mask.all():
                good_frames = np.where(~low_vis_mask)[0]
                bad_frames = np.where(low_vis_mask)[0]
                for axis in range(2):
                    good_vals = keypoints_seq[good_frames, kp_idx, axis]
                    interp_vals = np.interp(bad_frames, good_frames, good_vals)
                    keypoints_seq[bad_frames, kp_idx, axis] = interp_vals

        # --- Step 2: VISTA-style torso normalization ---
        kp_norm = self._normalize_keypoints(keypoints_seq)

        # --- Step 3: Keypoint position features (temporal statistics) ---
        kp_flat = kp_norm.reshape(kp_norm.shape[0], -1)  # (frames, 28)
        pos_feats = self._temporal_statistics(kp_flat)

        # --- Step 4: Keypoint velocity features ---
        if kp_flat.shape[0] > 1:
            velocity = np.diff(kp_flat, axis=0)
            velocity = np.vstack([velocity, velocity[-1:]])
        else:
            velocity = np.zeros_like(kp_flat)
        vel_feats = self._temporal_statistics(velocity)

        # --- Step 5: Keypoint acceleration features ---
        if velocity.shape[0] > 1:
            accel = np.diff(velocity, axis=0)
            accel = np.vstack([accel, accel[-1:]])
        else:
            accel = np.zeros_like(velocity)
        acc_feats = self._temporal_statistics(accel)

        # --- Step 6: Pairwise keypoint distance features ---
        distances = []
        for j1, j2 in self.KEYPOINT_PAIRS:
            dist = np.linalg.norm(kp_norm[:, j1, :] - kp_norm[:, j2, :], axis=1)
            distances.append(dist)
        dist_arr = np.array(distances).T  # (frames, n_pairs)
        dist_feats = self._temporal_statistics(dist_arr)

        # --- Step 7: Optical flow features ---
        flow_data = self._compute_optical_flow(frames)
        if flow_data.shape[0] > 0:
            flow_feats = self._temporal_statistics(flow_data)
        else:
            flow_feats = np.zeros(10 * 9)

        # --- Step 8: Color histogram features ---
        color_data = self._color_histogram_features(frames, bboxes)
        # Aggregate across frames
        color_agg = []
        for col in range(color_data.shape[1]):
            col_vals = color_data[:, col]
            color_agg.extend([np.mean(col_vals), np.std(col_vals)])
        color_feats = np.array(color_agg)

        # --- Step 9: Covariance of joint trajectories (MSDFE-inspired) ---
        if kp_flat.shape[0] > 1:
            cov_mat = np.cov(kp_flat.T)
            cov_feats = cov_mat[np.triu_indices(cov_mat.shape[0])]
        else:
            n = kp_flat.shape[1]
            cov_feats = np.zeros(n * (n + 1) // 2)

        # --- Concatenate all video features ---
        all_feats = np.concatenate([
            pos_feats,      # Keypoint position statistics
            vel_feats,      # Keypoint velocity statistics
            acc_feats,      # Keypoint acceleration statistics
            dist_feats,     # Pairwise distance statistics
            flow_feats,     # Optical flow statistics
            color_feats,    # Color histogram features
            cov_feats,      # Joint trajectory covariance
        ])

        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)

In [11]:
# =============================================================================
# SECTION 6: MULTIMODAL FUSION & CLASSIFICATION PIPELINE
# =============================================================================

class MultimodalHARPipeline:
    """
    Complete pipeline: load data, extract features, fuse, classify.
    """

    def __init__(self, config):
        self.config = config
        self.loader = UTDMHADLoader(config)
        self.skel_extractor = SkeletonFeatureExtractor()
        self.iner_extractor = InertialFeatureExtractor(config)
        self.depth_extractor = DepthFeatureExtractor(config)
        self.video_extractor = VideoFeatureExtractor(config)
        self.scaler = StandardScaler()

    def _build_sample_key(self, action, subject, trial):
        return f"a{action}_s{subject}_t{trial}"

    def load_all_data(self):
        """Load all modalities and organize by sample key."""
        samples = {}

        if self.config.USE_SKELETON:
            skel_data = self.loader.load_skeleton_data()
            for action, subject, trial, data in skel_data:
                key = self._build_sample_key(action, subject, trial)
                if key not in samples:
                    samples[key] = {'action': action, 'subject': subject, 'trial': trial}
                samples[key]['skeleton'] = data

        if self.config.USE_INERTIAL:
            iner_data = self.loader.load_inertial_data()
            for action, subject, trial, data in iner_data:
                key = self._build_sample_key(action, subject, trial)
                if key not in samples:
                    samples[key] = {'action': action, 'subject': subject, 'trial': trial}
                samples[key]['inertial'] = data

        if self.config.USE_DEPTH:
            depth_data = self.loader.load_depth_data()
            for action, subject, trial, data in depth_data:
                key = self._build_sample_key(action, subject, trial)
                if key not in samples:
                    samples[key] = {'action': action, 'subject': subject, 'trial': trial}
                samples[key]['depth'] = data

        if self.config.USE_VIDEO:
            video_data = self.loader.load_video_data()
            for action, subject, trial, data in video_data:
                key = self._build_sample_key(action, subject, trial)
                if key not in samples:
                    samples[key] = {'action': action, 'subject': subject, 'trial': trial}
                samples[key]['video'] = data

        logger.info(f"Total unique samples: {len(samples)}")
        return samples

    def extract_features(self, samples):
        """
        Extract features from all samples across all modalities.

        Returns:
            features_dict: {key: {'features': array, 'label': int, 'subject': int}}
        """
        features_dict = {}
        total = len(samples)

        for i, (key, sample) in enumerate(samples.items()):
            if (i + 1) % 50 == 0 or i == 0:
                logger.info(f"Extracting features: {i+1}/{total}")

            feature_parts = []

            # Skeleton features
            if self.config.USE_SKELETON and 'skeleton' in sample:
                skel_feats = self.skel_extractor.extract(sample['skeleton'])
                if skel_feats is not None:
                    feature_parts.append(('skeleton', skel_feats))

            # Inertial features
            if self.config.USE_INERTIAL and 'inertial' in sample:
                iner_feats = self.iner_extractor.extract(sample['inertial'])
                if iner_feats is not None:
                    feature_parts.append(('inertial', iner_feats))

            # Depth features
            if self.config.USE_DEPTH and 'depth' in sample:
                depth_feats = self.depth_extractor.extract(sample['depth'])
                if depth_feats is not None:
                    feature_parts.append(('depth', depth_feats))

            # Video (RGB) features
            if self.config.USE_VIDEO and 'video' in sample:
                video_feats = self.video_extractor.extract(
                    sample['video'],
                    apply_zeroing=True,
                    zero_fraction=ZERO_FRACTION,
                    zero_rng=np.random.RandomState(SEED + i),
                )
                if video_feats is not None:
                    feature_parts.append(('video', video_feats))

            if feature_parts:
                # Concatenate all modality features (feature-level fusion)
                concatenated = np.concatenate([f for _, f in feature_parts])
                features_dict[key] = {
                    'features': concatenated,
                    'label': sample['action'],
                    'subject': sample['subject'],
                    'modalities': [name for name, _ in feature_parts],
                    'modality_sizes': {name: len(f) for name, f in feature_parts},
                    'modality_features': {name: f for name, f in feature_parts},
                }

        logger.info(f"Extracted features for {len(features_dict)} samples")

        # Log feature dimensions
        if features_dict:
            sample_entry = next(iter(features_dict.values()))
            logger.info(f"Total feature vector dimension: {len(sample_entry['features'])}")
            for mod, size in sample_entry['modality_sizes'].items():
                logger.info(f"  {mod}: {size} features")

        return features_dict


    def save_features(self, features_dict):
        """
        Save extracted features for later use (e.g., subject-wise k-fold CV).

        Saves:
          - X_feat.pkl: list of dicts, each with 'depth_feat', 'sensor_feat',
                        'skeleton_feat', 'video_feat' numpy arrays
          - y.npy: encoded action labels (via LabelEncoder)
          - subjects.npy: subject IDs per sample
          - label_encoder.pkl: fitted LabelEncoder
        """
        out_dir = self.config.FEATURES_DIR
        os.makedirs(out_dir, exist_ok=True)

        # Build aligned list sorted by sample key
        keys = sorted(features_dict.keys())

        X_feat = []
        y_raw = []
        subjects = []

        for k in keys:
            entry = features_dict[k]
            mod_feats = entry.get('modality_features', {})

            sample_dict = {
                'skeleton_feat': mod_feats.get('skeleton', np.array([])),
                'sensor_feat':   mod_feats.get('inertial', np.array([])),
                'depth_feat':    mod_feats.get('depth', np.array([])),
                'video_feat':    mod_feats.get('video', np.array([])),
            }
            X_feat.append(sample_dict)
            y_raw.append(entry['label'])
            subjects.append(entry['subject'])

        y_raw = np.array(y_raw)
        subjects = np.array(subjects)

        # Fit label encoder on action labels
        le = LabelEncoder()
        y = le.fit_transform(y_raw)

        # Save using joblib
        joblib.dump(X_feat, os.path.join(out_dir, 'X_feat.pkl'))
        np.save(os.path.join(out_dir, 'y.npy'), y)
        np.save(os.path.join(out_dir, 'subjects.npy'), subjects)
        joblib.dump(le, os.path.join(out_dir, 'label_encoder.pkl'))

        # Log summary
        logger.info(f"Features saved to '{out_dir}/':")
        logger.info(f"  X_feat.pkl:        {len(X_feat)} samples (list of dicts)")
        first = X_feat[0]
        for mod_key in ['skeleton_feat', 'sensor_feat', 'depth_feat', 'video_feat']:
            dim = first[mod_key].shape[0] if first[mod_key].size > 0 else 0
            logger.info(f"    {mod_key}: {dim} features")
        total = sum(first[m].shape[0] for m in first if first[m].size > 0)
        logger.info(f"    TOTAL: {total} features")
        logger.info(f"  y.npy:             {y.shape} (encoded, {len(le.classes_)} classes)")
        logger.info(f"  subjects.npy:      {subjects.shape} "
                     f"(subjects: {sorted(np.unique(subjects).tolist())})")
        logger.info(f"  label_encoder.pkl: classes = {le.classes_.tolist()}")

    def split_train_test(self, features_dict):
        """Split into train/test using the standard subject-based protocol."""
        X_train, y_train = [], []
        X_test, y_test = [], []

        for key, entry in features_dict.items():
            if entry['subject'] in self.config.TRAIN_SUBJECTS:
                X_train.append(entry['features'])
                y_train.append(entry['label'])
            elif entry['subject'] in self.config.TEST_SUBJECTS:
                X_test.append(entry['features'])
                y_test.append(entry['label'])

        X_train = np.array(X_train)
        y_train = np.array(y_train)
        X_test = np.array(X_test)
        y_test = np.array(y_test)

        logger.info(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
        logger.info(f"Feature dimension: {X_train.shape[1]}")
        logger.info(f"Number of classes: {len(np.unique(y_train))}")

        return X_train, y_train, X_test, y_test

    def train_and_evaluate(self, X_train, y_train, X_test, y_test):
        """Train Random Forest and evaluate."""

        # Standardize features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        # Handle any remaining NaN/Inf after scaling
        X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
        X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

        # Build Random Forest classifier
        rf = RandomForestClassifier(
            n_estimators=self.config.RF_N_ESTIMATORS,
            max_depth=self.config.RF_MAX_DEPTH,
            min_samples_split=self.config.RF_MIN_SAMPLES_SPLIT,
            min_samples_leaf=self.config.RF_MIN_SAMPLES_LEAF,
            random_state=self.config.RF_RANDOM_STATE,
            n_jobs=self.config.RF_N_JOBS,
            class_weight='balanced',
        )

        logger.info("Training Random Forest classifier...")
        start_time = time.time()
        rf.fit(X_train_scaled, y_train)
        train_time = time.time() - start_time
        logger.info(f"Training completed in {train_time:.2f} seconds")

        # Predict
        y_pred = rf.predict(X_test_scaled)

        # Evaluate
        accuracy = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro')
        f1_weighted = f1_score(y_test, y_pred, average='weighted')

        logger.info(f"\n{'='*60}")
        logger.info(f"RESULTS")
        logger.info(f"{'='*60}")
        logger.info(f"Overall Accuracy: {accuracy*100:.2f}%")
        logger.info(f"Macro F1-Score:   {f1_macro*100:.2f}%")
        logger.info(f"Weighted F1-Score: {f1_weighted*100:.2f}%")

        # Per-class report
        action_names = [f"Action {i}" for i in range(1, self.config.NUM_ACTIONS + 1)]
        unique_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
        target_names = [action_names[l-1] for l in unique_labels]

        report = classification_report(y_test, y_pred, labels=unique_labels,
                                        target_names=target_names, digits=3)
        logger.info(f"\nClassification Report:\n{report}")

        # Cross-validation on training set
        logger.info("Running 5-fold cross-validation on training set...")
        cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=5, scoring='accuracy')
        logger.info(f"CV Accuracy: {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")

        # Feature importance analysis
        importances = rf.feature_importances_
        top_k = 20
        top_indices = np.argsort(importances)[-top_k:][::-1]
        logger.info(f"\nTop {top_k} most important features (by index):")
        for idx in top_indices:
            logger.info(f"  Feature {idx}: importance = {importances[idx]:.4f}")

        return {
            'accuracy': accuracy,
            'f1_macro': f1_macro,
            'f1_weighted': f1_weighted,
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
            'train_time': train_time,
            'model': rf,
            'y_pred': y_pred,
        }

    def run(self):
        """Execute the full pipeline."""
        logger.info("="*60)
        logger.info("MULTIMODAL HAR PIPELINE FOR UTD-MHAD")
        logger.info("="*60)
        logger.info(f"Modalities: Skeleton={self.config.USE_SKELETON}, "
                     f"Inertial={self.config.USE_INERTIAL}, "
                     f"Depth={self.config.USE_DEPTH}, "
                     f"Video={self.config.USE_VIDEO}")
        logger.info(f"Train subjects: {self.config.TRAIN_SUBJECTS}")
        logger.info(f"Test subjects:  {self.config.TEST_SUBJECTS}")

        # Step 1: Load data
        logger.info("\n--- Step 1: Loading data ---")
        samples = self.load_all_data()

        if not samples:
            logger.error("No data loaded! Please check the dataset path.")
            logger.info(f"Expected path: {self.config.DATASET_ROOT}")
            logger.info("Download UTD-MHAD from: https://personal.utdallas.edu/~kehtar/UTD-MHAD.html")
            return None

        # Step 1b: Apply structured data zeroing (10% feature loss simulation)
        logger.info("\n--- Step 1b: Applying structured data zeroing (25%) ---")
        samples = apply_structured_zeroing_to_samples(samples)

        # Step 2: Extract features
        logger.info("\n--- Step 2: Extracting features ---")
        features_dict = self.extract_features(samples)

        if not features_dict:
            logger.error("No features extracted!")
            return None

        # Step 2b: Save features for offline use (e.g., subject-wise k-fold CV)
        logger.info("\n--- Step 2b: Saving features ---")
        self.save_features(features_dict)

        # Step 3: Split data
        logger.info("\n--- Step 3: Train/test split ---")
        X_train, y_train, X_test, y_test = self.split_train_test(features_dict)

        if X_train.shape[0] == 0 or X_test.shape[0] == 0:
            logger.error("Empty train or test set!")
            return None

        # Step 4: Train and evaluate
        logger.info("\n--- Step 4: Training and evaluation ---")
        results = self.train_and_evaluate(X_train, y_train, X_test, y_test)

        return results


In [12]:
# =============================================================================
# SECTION 7: DEMO MODE (generates synthetic data for testing)
# =============================================================================

def create_synthetic_dataset(root_dir, n_actions=27, n_subjects=8, n_trials=4):
    """
    Create a synthetic UTD-MHAD-like dataset for testing the pipeline.
    This generates random data with the correct structure and file naming.
    """
    logger.info("Creating synthetic dataset for demonstration...")

    for modality, subdir in [("skeleton", "Skeleton"), ("inertial", "Inertial"),
                              ("depth", "Depth")]:
        mod_dir = os.path.join(root_dir, subdir)
        os.makedirs(mod_dir, exist_ok=True)

        for a in range(1, n_actions + 1):
            for s in range(1, n_subjects + 1):
                for t in range(1, n_trials + 1):
                    fname = f"a{a}_s{s}_t{t}_{modality}.mat"
                    fpath = os.path.join(mod_dir, fname)

                    if os.path.exists(fpath):
                        continue

                    n_frames = np.random.randint(30, 120)

                    if modality == "skeleton":
                        # (frames, 20, 3) with class-dependent patterns
                        base = np.random.randn(n_frames, 20, 3) * 0.1
                        # Add action-specific signal
                        base[:, :, 0] += np.sin(np.linspace(0, a * np.pi, n_frames))[:, None]
                        base[:, :, 1] += np.cos(np.linspace(0, a * 0.5 * np.pi, n_frames))[:, None]
                        from scipy.io import savemat
                        savemat(fpath, {'d_skel': base.reshape(n_frames, 60)})

                    elif modality == "inertial":
                        # (samples, 6) with class-dependent patterns
                        n_samples = np.random.randint(100, 500)
                        data = np.random.randn(n_samples, 6) * 0.5
                        # Action-specific frequency
                        freq = a * 0.5
                        t_axis = np.linspace(0, 2 * np.pi * freq, n_samples)
                        data[:, 0] += np.sin(t_axis)
                        data[:, 3] += np.cos(t_axis * 0.7)
                        from scipy.io import savemat
                        savemat(fpath, {'d_iner': data})

                    elif modality == "depth":
                        # (height, width, frames) - small for speed
                        h, w = 60, 80
                        n_depth_frames = min(n_frames, 20)
                        depth = np.zeros((h, w, n_depth_frames))
                        for fr in range(n_depth_frames):
                            # Create a blob that moves based on action class
                            cy = h // 2 + int(5 * np.sin(a * fr / 10.0))
                            cx = w // 2 + int(5 * np.cos(a * fr / 10.0))
                            Y, X = np.ogrid[:h, :w]
                            mask = (Y - cy)**2 + (X - cx)**2 < (10 + a)**2
                            depth[:, :, fr][mask] = 1000 + a * 50 + np.random.rand() * 100
                        from scipy.io import savemat
                        savemat(fpath, {'d_depth': depth})

    logger.info(f"Synthetic dataset created at {root_dir}")


# =============================================================================
# SECTION 8: MAIN EXECUTION
# =============================================================================

def main():
    """Main entry point."""

    config = Config()

    # Check if real dataset exists
    if not os.path.isdir(config.DATASET_ROOT):
        logger.info(f"Dataset not found at '{config.DATASET_ROOT}'")
        logger.info("Creating synthetic dataset for demonstration...")
        os.makedirs(config.DATASET_ROOT, exist_ok=True)
        create_synthetic_dataset(config.DATASET_ROOT)

    # Run pipeline
    pipeline = MultimodalHARPipeline(config)
    results = pipeline.run()

    if results:
        logger.info(f"\n{'='*60}")
        logger.info("FINAL SUMMARY")
        logger.info(f"{'='*60}")
        logger.info(f"Accuracy:         {results['accuracy']*100:.2f}%")
        logger.info(f"Macro F1:         {results['f1_macro']*100:.2f}%")
        logger.info(f"Weighted F1:      {results['f1_weighted']*100:.2f}%")
        logger.info(f"CV Accuracy:      {results['cv_mean']*100:.2f}% "
                     f"(+/- {results['cv_std']*100:.2f}%)")
        logger.info(f"Training Time:    {results['train_time']:.2f}s")


if __name__ == "__main__":
    main()


I0000 00:00:1772962513.370435 3382758 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1772962513.466632 3382908 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 590.48.01), renderer: NVIDIA GeForce RTX 4090/PCIe/SSE2
2026-03-08 14:35:13,467 - INFO - ============================================================
2026-03-08 14:35:13,468 - INFO - MULTIMODAL HAR PIPELINE FOR UTD-MHAD
2026-03-08 14:35:13,468 - INFO - ============================================================
2026-03-08 14:35:13,468 - INFO - Modalities: Skeleton=True, Inertial=True, Depth=True, Video=True
2026-03-08 14:35:13,468 - INFO - Train subjects: [1, 3, 5, 7]
2026-03-08 14:35:13,468 - INFO - Test subjects:  [2, 4, 6, 8]
2026-03-08 14:35:13,469 - INFO - 
--- Step 1: Loading data ---
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1772962513.510781 3382878 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference.